# Final Audit Notebook (Based on `main_experiment_grid_search_0116_v5.ipynb`)

이 노트북은 **v5 실험이 의도대로 수행되었는지 최종 점검**하기 위한 리포트용 검증 노트북입니다.

검증 항목:
1. MovieLens fold 데이터(split 파일) 존재/형태 확인
2. Train_inner / Validation / Test 간 누수(leakage) 여부 확인
3. 결과 파일의 실험 조건(v5 설정) 일치 여부 확인
4. 정규화 패널티(λ=0.002) 공식 적용 여부 확인
5. 일부 실제 데이터/결과 샘플 표를 눈으로 확인

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

# --- v5 기준 실험 설정 (main_experiment_grid_search_0116_v5.ipynb) ---
FOLDS_TO_RUN = [8, 9, 10]
K_VALUES_EXPECTED = list(range(10, 101, 10))
TOPN_VALUES_EXPECTED = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
REGULARIZATION_LAMBDA = 0.002
METHODS_EXPECTED_COUNT = 17

root = Path.cwd().resolve()
if root.name == 'notebooks':
    project_root = root.parent
else:
    project_root = root

data_root = project_root / 'data' / 'movielenz_data'
results_root = project_root / 'results'

print('Project root:', project_root)
print('Data root   :', data_root)
print('Results root:', results_root)
print('v5 Folds    :', FOLDS_TO_RUN)
print('v5 Lambda   :', REGULARIZATION_LAMBDA)

Project root: D:\Dropbox\python_workspace(백업)\imputation_project
Data root   : D:\Dropbox\python_workspace(백업)\imputation_project\data\movielenz_data
Results root: D:\Dropbox\python_workspace(백업)\imputation_project\results
v5 Folds    : [8, 9, 10]
v5 Lambda   : 0.002


In [10]:
# 1) fold별 split 파일 존재 및 shape 확인
required_split_files = ['train_inner.csv', 'validation.csv', 'train.csv', 'test.csv']
rows = []
split_dfs = {}

for fold in FOLDS_TO_RUN:
    fold_id = f'fold_{fold:02d}'
    fold_dir = data_root / fold_id
    split_dfs[fold] = {}

    for fname in required_split_files:
        fpath = fold_dir / fname
        exists = fpath.exists()

        if exists:
            df = pd.read_csv(fpath, index_col=0)
            split_dfs[fold][fname] = df
            rows.append({
                'fold': fold,
                'file': fname,
                'exists': True,
                'shape': f'{df.shape[0]}x{df.shape[1]}',
                'observed_cells': int(df.notna().sum().sum()),
                'sparsity_%': round(100 * (1 - df.notna().sum().sum() / df.size), 2)
            })
        else:
            rows.append({
                'fold': fold,
                'file': fname,
                'exists': False,
                'shape': None,
                'observed_cells': None,
                'sparsity_%': None
            })

split_status = pd.DataFrame(rows).sort_values(['fold', 'file']).reset_index(drop=True)
display(split_status)

missing = split_status[~split_status['exists']]
if len(missing) == 0:
    print('✅ 모든 fold에 필요한 split 파일이 존재합니다.')
else:
    print('❌ 누락된 파일이 있습니다.')
    display(missing)

,fold,file,exists,shape,observed_cells,sparsity_%
0,8,test.csv,True,1682x943,9738,99.39
1,8,train.csv,True,1682x943,90262,94.31
2,8,train_inner.csv,True,1682x943,72189,95.45
3,8,validation.csv,True,1682x943,18073,98.86
4,9,test.csv,True,1682x943,9652,99.39
5,9,train.csv,True,1682x943,90348,94.30
6,9,train_inner.csv,True,1682x943,72259,95.44
7,9,validation.csv,True,1682x943,18089,98.86
8,10,test.csv,True,1682x943,9587,99.40
9,10,train.csv,True,1682x943,90413,94.30


✅ 모든 fold에 필요한 split 파일이 존재합니다.


In [11]:
# 2) 데이터 누수(leakage) 점검: observed 좌표 중복 여부
def observed_coords(df: pd.DataFrame):
    arr = ~df.isna().values
    coords = np.argwhere(arr)
    return set((int(r), int(c)) for r, c in coords)

leakage_rows = []
for fold in FOLDS_TO_RUN:
    d = split_dfs[fold]
    train_inner = d['train_inner.csv']
    val = d['validation.csv']
    train_full = d['train.csv']
    test = d['test.csv']

    c_train_inner = observed_coords(train_inner)
    c_val = observed_coords(val)
    c_train_full = observed_coords(train_full)
    c_test = observed_coords(test)

    overlap_inner_val = len(c_train_inner & c_val)
    overlap_inner_test = len(c_train_inner & c_test)
    overlap_val_test = len(c_val & c_test)

    # train_full은 train_inner + validation의 합이어야 함
    union_inner_val = c_train_inner | c_val
    train_full_extra = len(c_train_full - union_inner_val)
    train_full_missing = len(union_inner_val - c_train_full)

    leakage_rows.append({
        'fold': fold,
        'overlap(train_inner, validation)': overlap_inner_val,
        'overlap(train_inner, test)': overlap_inner_test,
        'overlap(validation, test)': overlap_val_test,
        'train_full_extra_cells': train_full_extra,
        'train_full_missing_cells': train_full_missing
    })

leakage_df = pd.DataFrame(leakage_rows)
display(leakage_df)

ok = (
    (leakage_df['overlap(train_inner, validation)'] == 0).all() and
    (leakage_df['overlap(train_inner, test)'] == 0).all() and
    (leakage_df['overlap(validation, test)'] == 0).all() and
    (leakage_df['train_full_extra_cells'] == 0).all() and
    (leakage_df['train_full_missing_cells'] == 0).all()
)

print('✅ Leakage 점검 통과' if ok else '❌ Leakage 의심 항목 존재')

,fold,"overlap(train_inner, validation)","overlap(train_inner, test)","overlap(validation, test)",train_full_extra_cells,train_full_missing_cells
0,8,0,0,0,0,0
1,9,0,0,0,0,0
2,10,0,0,0,0,0


✅ Leakage 점검 통과


In [12]:
# 3) 결과 파일 로딩 및 v5 실험 조건 일치 여부 점검
result_files = {
    8: results_root / 'fold_08' / 'grid_search_results_20260128_222923.csv',
    9: results_root / 'fold_09' / 'grid_search_results_20260129_023436.csv',
    10: results_root / 'fold_10' / 'grid_search_results_20260129_063658.csv',
}

result_dfs = {}
summary_rows = []

for fold, path in result_files.items():
    if not path.exists():
        summary_rows.append({
            'fold': fold,
            'file_exists': False,
            'n_rows': None,
            'n_methods': None,
            'k_values_ok': None,
            'topn_values_ok': None,
            'type_values': None
        })
        continue

    df = pd.read_csv(path)
    result_dfs[fold] = df

    methods = sorted(df['method'].dropna().unique().tolist())
    k_values = sorted(df['K'].dropna().unique().tolist())
    topn_values = sorted(df['TopN'].dropna().unique().tolist())
    type_values = sorted(df['type'].dropna().unique().tolist())

    summary_rows.append({
        'fold': fold,
        'file_exists': True,
        'n_rows': int(len(df)),
        'n_methods': int(len(methods)),
        'k_values_ok': (k_values == K_VALUES_EXPECTED),
        'topn_values_ok': (topn_values == TOPN_VALUES_EXPECTED),
        'type_values': ','.join(type_values)
    })

result_summary = pd.DataFrame(summary_rows).sort_values('fold')
display(result_summary)

all_ok = (
    result_summary['file_exists'].all() and
    (result_summary['n_methods'] == METHODS_EXPECTED_COUNT).all() and
    result_summary['k_values_ok'].all() and
    result_summary['topn_values_ok'].all() and
    result_summary['type_values'].str.contains('baseline').all() and
    result_summary['type_values'].str.contains('optimized').all()
)

print('✅ v5 설정 일치 확인' if all_ok else '⚠️ 일부 설정 불일치 가능')

,fold,file_exists,n_rows,n_methods,k_values_ok,topn_values_ok,type_values
0,8,True,3400,17,True,True,"baseline,optimized"
1,9,True,3400,17,True,True,"baseline,optimized"
2,10,True,3400,17,True,True,"baseline,optimized"


✅ v5 설정 일치 확인


In [13]:
# 4) 정규화 패널티 공식 검증: penalty = lambda * |alpha - 1|
penalty_checks = []

for fold, df in result_dfs.items():
    # optimized
    d_opt = df[df['type'] == 'optimized'].copy()
    expected_penalty = REGULARIZATION_LAMBDA * (d_opt['alpha'] - 1.0).abs()
    diff = (d_opt['regularization_penalty'] - expected_penalty).abs()
    max_abs_diff = float(diff.max()) if len(diff) else np.nan
    pass_rate = float((diff < 1e-9).mean()) if len(diff) else np.nan

    # baseline
    d_base = df[df['type'] == 'baseline'].copy()
    baseline_penalty_ok = bool((d_base['regularization_penalty'].fillna(0) == 0).all())

    penalty_checks.append({
        'fold': fold,
        'optimized_rows': int(len(d_opt)),
        'max_abs_diff_penalty': max_abs_diff,
        'optimized_exact_match_rate': round(pass_rate, 4),
        'baseline_penalty_all_zero': baseline_penalty_ok
    })

penalty_df = pd.DataFrame(penalty_checks).sort_values('fold')
display(penalty_df)

penalty_ok = (
    (penalty_df['max_abs_diff_penalty'] < 1e-9).all() and
    penalty_df['baseline_penalty_all_zero'].all()
)
print('✅ 정규화 패널티 공식 일치' if penalty_ok else '⚠️ 패널티 계산 재확인 필요')

,fold,optimized_rows,max_abs_diff_penalty,optimized_exact_match_rate,baseline_penalty_all_zero
0,8,1700,9.985502e-17,1.0,True
1,9,1700,9.985502e-17,1.0,True
2,10,1700,9.985502e-17,1.0,True


✅ 정규화 패널티 공식 일치


In [14]:
# 5) 눈으로 확인할 수 있는 샘플 데이터 (Fold 08)
fold = 8
print('=== Fold 08: split 데이터 샘플 (상위 5행, 6열) ===')
for fname in required_split_files:
    df = split_dfs[fold][fname]
    print(f'\n[{fname}] shape={df.shape}, observed={int(df.notna().sum().sum())}')
    display(df.iloc[:5, :6])

=== Fold 08: split 데이터 샘플 (상위 5행, 6열) ===

[train_inner.csv] shape=(1682, 943), observed=72189


,1,2,3,4,5,6
1,5.0,4.0,NaN,NaN,4.0,4.0
2,3.0,NaN,NaN,NaN,3.0,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN
4,3.0,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN



[validation.csv] shape=(1682, 943), observed=18073


,1,2,3,4,5,6
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN



[train.csv] shape=(1682, 943), observed=90262


,1,2,3,4,5,6
item_id,,,,,,
1,5.0,4.0,NaN,NaN,4.0,4.0
2,3.0,NaN,NaN,NaN,3.0,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN
4,3.0,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN



[test.csv] shape=(1682, 943), observed=9738


,1,2,3,4,5,6
item_id,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
5,3.0,NaN,NaN,NaN,NaN,NaN


In [15]:
# 6) 눈으로 확인할 수 있는 결과 샘플 (Fold 08, method='pcc', K=20)
df8 = result_dfs[8]
sample = df8[(df8['method'] == 'pcc') & (df8['K'] == 20)]
sample_view = sample[['fold','method','type','alpha','K','TopN','validation_mse','regularization_penalty','test_RMSE','test_MAD','test_Precision','test_Recall']]
sample_view = sample_view.sort_values(['TopN','type'])
display(sample_view.head(20))

# baseline vs optimized 차이 요약 (fold 8, pcc, K=20)
pivot = sample.pivot_table(index='TopN', columns='type', values='test_RMSE', aggfunc='mean').reset_index()
pivot['delta_opt_minus_base'] = pivot['optimized'] - pivot['baseline']
display(pivot.head(10))

,fold,method,type,alpha,K,TopN,validation_mse,regularization_penalty,test_RMSE,test_MAD,test_Precision,test_Recall
30,8,pcc,baseline,1.0,20,5,NaN,0.000,1.023513,0.810405,0.673719,0.731904
20,8,pcc,optimized,3.0,20,5,0.687111,0.004,1.040430,0.820008,0.670113,0.730094
31,8,pcc,baseline,1.0,20,10,NaN,0.000,1.023513,0.810405,0.633738,0.890470
21,8,pcc,optimized,3.0,20,10,0.687111,0.004,1.040430,0.820008,0.630875,0.886860
32,8,pcc,baseline,1.0,20,15,NaN,0.000,1.023513,0.810405,0.607903,0.949365
22,8,pcc,optimized,3.0,20,15,0.687111,0.004,1.040430,0.820008,0.607337,0.948538
33,8,pcc,baseline,1.0,20,20,NaN,0.000,1.023513,0.810405,0.595709,0.976531
23,8,pcc,optimized,3.0,20,20,0.687111,0.004,1.040430,0.820008,0.595869,0.976784
34,8,pcc,baseline,1.0,20,25,NaN,0.000,1.023513,0.810405,0.589610,0.990530
24,8,pcc,optimized,3.0,20,25,0.687111,0.004,1.040430,0.820008,0.589313,0.989961


type,TopN,baseline,optimized,delta_opt_minus_base
0,5,1.023513,1.04043,0.016917
1,10,1.023513,1.04043,0.016917
2,15,1.023513,1.04043,0.016917
3,20,1.023513,1.04043,0.016917
4,25,1.023513,1.04043,0.016917
5,30,1.023513,1.04043,0.016917
6,35,1.023513,1.04043,0.016917
7,40,1.023513,1.04043,0.016917
8,45,1.023513,1.04043,0.016917
9,50,1.023513,1.04043,0.016917


In [16]:
# 7) 최종 PASS/FAIL 체크리스트
checks = pd.DataFrame([
    {'check': 'Split files exist', 'pass': len(split_status[~split_status['exists']]) == 0},
    {'check': 'No data leakage across split', 'pass': bool(ok)},
    {'check': 'v5 experiment conditions match', 'pass': bool(all_ok)},
    {'check': 'Penalty formula match (lambda=0.002)', 'pass': bool(penalty_ok)},
])
display(checks)

if checks['pass'].all():
    print('🎉 FINAL AUDIT RESULT: PASS')
    print('v5 기준으로 MovieLens 실험 파이프라인이 의도대로 수행된 것으로 판단됩니다.')
else:
    print('❌ FINAL AUDIT RESULT: ATTENTION NEEDED')
    print('실패한 항목을 확인 후 교수님 보고 전 재검토가 필요합니다.')

,check,pass
0,Split files exist,True
1,No data leakage across split,True
2,v5 experiment conditions match,True
3,Penalty formula match (lambda=0.002),True


🎉 FINAL AUDIT RESULT: PASS
v5 기준으로 MovieLens 실험 파이프라인이 의도대로 수행된 것으로 판단됩니다.
